# 02 — Categorical encoding and feature selection

Airport and airline codes/names are nominal categories, not measurements. We demonstrate LabelEncoder as a compact way to turn a target label into numbers, and show why OneHotEncoder is safer for nominal input features: one-hot codes do not imply that airport 9 is “greater” than airport 3.

We then compare three feature-selection approaches on a reproducible sample of the actual passenger dataset.The carrier name is descriptive metadata duplicated by the carrier code; the encoding experiment one-hot encodes every text category (including the name), while prediction pipelines use only the code to avoid duplicate inputs and keep the name for display. Numeric airport IDs are administrative identifiers and are not treated as measurements.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import f_regression, mutual_info_regression
from sklearn.ensemble import ExtraTreesRegressor

df = pd.read_csv(Path("data/processed/air_passenger_monthly.csv"))
df["DATE"] = pd.to_datetime(df.DATE)
df["LOG_PASSENGERS"] = np.log1p(df.PASSENGERS)
df["MONTH_SIN"] = np.sin(2*np.pi*df.MONTH/12)
df["MONTH_COS"] = np.cos(2*np.pi*df.MONTH/12)
df.shape, df.head()

((503224, 14),         DATE  YEAR  MONTH  ... LOG_PASSENGERS MONTH_SIN MONTH_COS
0 2025-01-01  2025      1  ...       1.386294       0.5  0.866025
1 2025-01-01  2025      1  ...       0.000000       0.5  0.866025
2 2025-01-01  2025      1  ...       1.791759       0.5  0.866025
3 2025-01-01  2025      1  ...       6.386879       0.5  0.866025
4 2025-01-01  2025      1  ...       0.693147       0.5  0.866025

[5 rows x 14 columns])


## Label encoding example
LabelEncoder sorts the two target labels alphabetically, so this demonstration maps High→0 and Normal→1. This is a class code, not a ranking. The separate classifier uses an explicit Normal=0, High=1 target so that positive-class recall means high-demand recall. For input airline names, integer codes are shown only to demonstrate the technique; use one-hot encoding for the actual nominal input columns.

In [2]:
threshold = df.loc[df.YEAR.eq(2025), "PASSENGERS"].quantile(.75)
df["DEMAND_LABEL"] = np.where(df.PASSENGERS >= threshold, "High", "Normal")
label_encoder = LabelEncoder()
df["DEMAND_LABEL_ENCODED"] = label_encoder.fit_transform(df.DEMAND_LABEL)
carrier_encoder = LabelEncoder()
df["CARRIER_LABEL_DEMO"] = carrier_encoder.fit_transform(df.UNIQUE_CARRIER.astype(str))
print("Demand classes:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("Example airline code mapping:", dict(zip(carrier_encoder.classes_[:8], carrier_encoder.transform(carrier_encoder.classes_[:8]))))
print("Caution: carrier integer values are category IDs, not airline rankings.")

Demand classes: {'High': np.int64(0), 'Normal': np.int64(1)}
Example airline code mapping: {'02Q': np.int64(0), '07Q': np.int64(1), '0BQ': np.int64(2), '0CQ': np.int64(3), '0J': np.int64(4), '0Q': np.int64(5), '0QQ': np.int64(6), '0UQ': np.int64(7)}
Caution: carrier integer values are category IDs, not airline rankings.


## One-hot encode route categories
To keep the classroom comparison quick and readable, we sample up to 40,000 records and retain the most frequent airports/carriers plus an “OTHER” group. A real production analysis could preserve every category with a sparse matrix.

In [3]:
selection_pool = df[df.YEAR.eq(2025)].copy()
sample = selection_pool.sample(n=min(40_000, len(selection_pool)), random_state=42).copy()
print("Feature ranking uses training-period (2025) rows only:", len(sample))
for col in ["ORIGIN", "DEST", "UNIQUE_CARRIER", "UNIQUE_CARRIER_NAME"]:
    popular = set(df[col].value_counts().head(12).index.astype(str))
    sample[col + "_GROUP"] = sample[col].astype(str).where(sample[col].astype(str).isin(popular), "OTHER")

cat_cols = ["ORIGIN_GROUP", "DEST_GROUP", "UNIQUE_CARRIER_GROUP", "UNIQUE_CARRIER_NAME_GROUP"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(sample[cat_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols), index=sample.index)
display(encoded_df.head())
print("Original category columns:", len(cat_cols), "| One-hot features:", encoded_df.shape[1])

Feature ranking uses training-period (2025) rows only: 40000
        ORIGIN_GROUP_ATL  ...  UNIQUE_CARRIER_NAME_GROUP_UNITED PARCEL SERVICE
277205               0.0  ...                                              0.0
294608               0.0  ...                                              0.0
122328               0.0  ...                                              0.0
100519               0.0  ...                                              0.0
310585               0.0  ...                                              0.0

[5 rows x 52 columns]
Original category columns: 4 | One-hot features: 52


## Feature-selection comparison
The target is log passenger count to reduce the effect of a few very large markets. We compare:
1. **F-test (filter):** scores a feature’s linear association with the target.
2. **Mutual information (filter):** can detect broader, non-linear dependence.
3. **Extra Trees importance (embedded):** ranks features used by an ensemble of decision trees.

The reduced category set is deliberate so the notebook remains practical on a student laptop. Feature ranks are evidence for discussion, not proof of causation.

In [4]:
X = pd.concat([
    sample[["DISTANCE", "MONTH_SIN", "MONTH_COS"]].fillna(0).reset_index(drop=True),
    encoded_df.reset_index(drop=True)
], axis=1)
y = sample.LOG_PASSENGERS.reset_index(drop=True)

f_scores, _ = f_regression(X, y)
mi_scores = mutual_info_regression(X, y, random_state=42)
tree = ExtraTreesRegressor(n_estimators=80, min_samples_leaf=5, n_jobs=-1, random_state=42)
tree.fit(X, y)

ranking = pd.DataFrame({
    "feature": X.columns,
    "F_score": f_scores,
    "Mutual_information": mi_scores,
    "ExtraTrees_importance": tree.feature_importances_
})
for score in ["F_score", "Mutual_information", "ExtraTrees_importance"]:
    print("\nTop features by", score)
    display(ranking.nlargest(10, score)[["feature", score]])


Top features by F_score
                                              feature      F_score
33                            UNIQUE_CARRIER_GROUP_FX  3907.130665
46  UNIQUE_CARRIER_NAME_GROUP_FEDERAL EXPRESS CORP...  3907.130665
29                            UNIQUE_CARRIER_GROUP_5X  2950.027089
54    UNIQUE_CARRIER_NAME_GROUP_UNITED PARCEL SERVICE  2950.027089
37                         UNIQUE_CARRIER_GROUP_OTHER  1961.034029
48                    UNIQUE_CARRIER_NAME_GROUP_OTHER  1961.034029
30                            UNIQUE_CARRIER_GROUP_AA  1818.965194
43   UNIQUE_CARRIER_NAME_GROUP_AMERICAN AIRLINES INC.  1818.965194
27                                   DEST_GROUP_OTHER  1669.324009
14                                 ORIGIN_GROUP_OTHER  1457.677555

Top features by Mutual_information
                                              feature  Mutual_information
0                                            DISTANCE            0.243449
46  UNIQUE_CARRIER_NAME_GROUP_FEDERAL EXPRESS CORP... 

### Takeaway
For route prediction, keep origin, destination, airline, distance, and calendar seasonality because they have a clear practical interpretation. Use one-hot encoding for nominal categories; avoid treating airport IDs or label-encoded airline codes as continuous numbers. Feature selection helps reduce clutter, but a feature must be selected using training data only when you build a predictive model.


## How many features survive selection?

The ranked list is useful for explanation, but the requirement also asks for a concrete before/after count. We use a top-10 F-test filter here as a compact teaching choice. In a real model, the number retained should be selected inside a training-only validation pipeline; rankings are associations, not causal evidence.


In [5]:
from sklearn.feature_selection import SelectKBest, f_regression
k_keep = min(10, X.shape[1])
selector = SelectKBest(score_func=f_regression, k=k_keep)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support()].tolist()
print(f"Feature selection: started with {X.shape[1]} encoded features; retained {X_selected.shape[1]} (top F-test).")
print("Retained:", selected_feature_names)


Feature selection: started with 55 encoded features; retained 10 (top F-test).
Retained: ['ORIGIN_GROUP_OTHER', 'DEST_GROUP_OTHER', 'UNIQUE_CARRIER_GROUP_5X', 'UNIQUE_CARRIER_GROUP_AA', 'UNIQUE_CARRIER_GROUP_FX', 'UNIQUE_CARRIER_GROUP_OTHER', 'UNIQUE_CARRIER_NAME_GROUP_AMERICAN AIRLINES INC.', 'UNIQUE_CARRIER_NAME_GROUP_FEDERAL EXPRESS CORPORATION', 'UNIQUE_CARRIER_NAME_GROUP_OTHER', 'UNIQUE_CARRIER_NAME_GROUP_UNITED PARCEL SERVICE']
